In [1]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt


import numpy as np
import pandas as pd
from scipy.stats import describe
pd.options.display.max_columns = 12
pd.options.display.max_rows = 24

# disable warnings in Anaconda
import warnings
warnings.simplefilter('ignore')

# plots inisde jupyter notebook
%matplotlib inline
import matplotlib.pyplot as plt

import seaborn as sns
sns.set(style='darkgrid', palette='muted')
color_scheme = {
    'red': '#F1637A',
    'green': '#6ABB3E',
    'blue': '#3D8DEA',
    'black': '#000000'
}



In [2]:
df_wm_sales_e=pd.read_csv('data/sales_train_evaluation.csv')

In [3]:
df_wm_sales_e.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1947 entries, id to d_1941
dtypes: int64(1941), object(6)
memory usage: 452.9+ MB


In [4]:
df_wm_sales_v=pd.read_csv('data/sales_train_validation.csv')
df_wm_sales_v.info() #validation has less day values (1942 vs 1913). so lets use evaluation data.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1919 entries, id to d_1913
dtypes: int64(1913), object(6)
memory usage: 446.4+ MB


In [5]:
df_wm_sales_e.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1',
       'd_2', 'd_3', 'd_4',
       ...
       'd_1932', 'd_1933', 'd_1934', 'd_1935', 'd_1936', 'd_1937', 'd_1938',
       'd_1939', 'd_1940', 'd_1941'],
      dtype='object', length=1947)

In [6]:
print(f'Number of items: {df_wm_sales_e["item_id"].nunique()}')
print(f'Number of departments: {df_wm_sales_e["dept_id"].nunique()}')
print(f'Number of categories: {df_wm_sales_e["cat_id"].nunique()}')
print(f'Number of stores: {df_wm_sales_e["store_id"].nunique()}')
print(f'Number of states: {df_wm_sales_e["state_id"].nunique()}')

Number of items: 3049
Number of departments: 7
Number of categories: 3
Number of stores: 10
Number of states: 3


In [7]:
df_wm_sales_e["cat_id"].value_counts()

cat_id
FOODS        14370
HOUSEHOLD    10470
HOBBIES       5650
Name: count, dtype: int64

In [8]:
df_wm_sales_e["state_id"].value_counts()

state_id
CA    12196
TX     9147
WI     9147
Name: count, dtype: int64

In [9]:
import pandas as pd

# 1. Load base data (wide format)

df_sales_wide = df_wm_sales_e.copy()


# 2. Filter to FOODS category

df_cat = df_sales_wide[df_sales_wide["cat_id"] == "FOODS"].copy()


# 3. Compute total sales across all days

day_cols = [c for c in df_cat.columns if c.startswith("d_")]
df_cat["total_sales"] = df_cat[day_cols].sum(axis=1)


# 4. Select TOP 150 items (largest total sales across all stores & states)

df_top150 = (
    df_cat
    .sort_values("total_sales", ascending=False)
    .head(150)
    .copy()
)

df_top150 = df_top150.drop(columns=["total_sales"])

print("Wide subset shape:", df_top150.shape)
print(df_top150["state_id"].value_counts())
print(df_top150["store_id"].nunique(), "stores included")


# 5. Melt to long format (safe)

df_long = df_top150.melt(
    id_vars=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

print("Long shape after melt:", df_long.shape)


# 6. Merge with calendar

calendar = pd.read_csv("data/calendar.csv")

df_merged = df_long.merge(calendar, on="d", how="left")

# 7. Merge with sell_price

sell_price = pd.read_csv("data/sell_prices.csv")

df_merged = df_merged.merge(
    sell_price,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)


# 8. Final clean-up

df_merged["date"] = pd.to_datetime(df_merged["date"])
df_merged = df_merged.sort_values(["item_id", "store_id", "date"])

print("Final dataset shape:", df_merged.shape)
#print(df_merged.head())


Wide subset shape: (150, 1947)
state_id
CA    67
WI    44
TX    39
Name: count, dtype: int64
10 stores included
Long shape after melt: (291150, 8)
Final dataset shape: (291150, 22)


In [10]:
df_merged.to_csv('data/wallmart_food_top150.csv')

In [11]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Index: 291150 entries, 145 to 291147
Data columns (total 22 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   id            291150 non-null  object        
 1   item_id       291150 non-null  object        
 2   dept_id       291150 non-null  object        
 3   cat_id        291150 non-null  object        
 4   store_id      291150 non-null  object        
 5   state_id      291150 non-null  object        
 6   d             291150 non-null  object        
 7   sales         291150 non-null  int64         
 8   date          291150 non-null  datetime64[ns]
 9   wm_yr_wk      291150 non-null  int64         
 10  weekday       291150 non-null  object        
 11  wday          291150 non-null  int64         
 12  month         291150 non-null  int64         
 13  year          291150 non-null  int64         
 14  event_name_1  23700 non-null   object        
 15  event_type_1  23700 

In [14]:
df=pd.read_csv('data/wallmart_food_top150.csv', parse_dates=['date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 291150 entries, 0 to 291149
Data columns (total 23 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Unnamed: 0    291150 non-null  int64         
 1   id            291150 non-null  object        
 2   item_id       291150 non-null  object        
 3   dept_id       291150 non-null  object        
 4   cat_id        291150 non-null  object        
 5   store_id      291150 non-null  object        
 6   state_id      291150 non-null  object        
 7   d             291150 non-null  object        
 8   sales         291150 non-null  int64         
 9   date          291150 non-null  datetime64[ns]
 10  wm_yr_wk      291150 non-null  int64         
 11  weekday       291150 non-null  object        
 12  wday          291150 non-null  int64         
 13  month         291150 non-null  int64         
 14  year          291150 non-null  int64         
 15  event_name_1  237

In [ ]:
df['date'].min()

In [ ]:
df.head()

In [ ]:
print(f'Number of items: {df["item_id"].nunique()}')
print(f'Number of departments: {df["dept_id"].nunique()}')
print(f'Number of categories: {df["cat_id"].nunique()}')
print(f'Number of stores: {df["store_id"].nunique()}')
print(f'Number of states: {df["state_id"].nunique()}')

In [ ]:
df.shape

- items sold everday
- check null values for single item
- how often prices change
- clean data
- milestone for each week
---
- For demand + pricing model:
- analysis/modelling level: item_id × store_id × date

In [ ]:
df_item1=df[df['item_id']=='FOODS_1_096']

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
df_item1 = df_item1.sort_values("date")
df_item1 = df_item1.dropna(subset=["sell_price"])


plt.figure(figsize=(12,5))
plt.plot(df_item1["date"], df_item1["sell_price"], drawstyle="steps-post")
plt.title("Sell Price Over Time")
plt.xlabel("Date")
plt.ylabel("Sell Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df_item1["price_change"] = df_item1["sell_price"].diff().abs()

num_changes = (df_item1["price_change"] > 0).sum()
total_days = len(df_item1)

print("Price changes:", num_changes)
print("Total days:", total_days)
print("Change frequency:", num_changes / total_days)

In [ ]:
df.groupby("item_id")["sell_price"].nunique().describe()

In [ ]:
sns.lineplot(data=df_item1, x="date", y='sales')
plt.show()

In [15]:
calendar = pd.read_csv("data/calendar.csv")
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [ ]:
df.isna().sum()

In [ ]:
sell_prices=pd.read_csv('data/sell_prices.csv')

In [ ]:
sell_prices.info()

In [ ]:
sell_prices.isna().sum()